# 2.2 - Modeling - Fine Tunning

Esta etapa é uma continuação da anterior, **2.1 - Modeling.ipynb**, que foi utilizada para se definir o melhor modelo para a análise em questão com base nos quesitos: tempo, recall e xxx (test_average_precision).

A partir de agora será efetuado o **Fine Tunning** para obtermos o melhor modelo, então exportar este utilizando a biblioteca `joblib` e então, aplicar aos testes na seção `2.3 - Modeling_Test`

## Importing Libs

In [1]:
# Importing Libs
    
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="darkgrid", rc={'figure.figsize':(10,6)})

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report,accuracy_score, recall_score

from xgboost import XGBClassifier

import warnings

## Importing Dataset and Feature Engeneering

Nesta etapa:
- Foi importado o dataset
- Foi utilizdo os valores criados na seção **1 - EDA.ipynb**, `balanceOrigDiff` e `balanceDestDiff`
- Foram removidos valores que não seriam relevanesna previsão

In [3]:
FRAUD_PATH = "datasets/AIML Dataset.csv"

df = pd.read_csv(FRAUD_PATH)

categorical_features = ['type', 'nameOrig', 'nameDest', 'isFraud', 'isFlaggedFraud']
numerical_features = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']

df['balanceOrigDiff'] = df['newbalanceOrig'] - df['oldbalanceOrg']
df['balanceDestDiff'] = df['newbalanceDest'] - df['oldbalanceDest']

features_model = numerical_features + ['balanceOrigDiff', 'balanceDestDiff', 'isFraud']
df_model = df[features_model]

df_model.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,balanceOrigDiff,balanceDestDiff,isFraud
0,1,9839.64,170136.0,160296.36,0.0,0.0,-9839.64,0.0,0
1,1,1864.28,21249.0,19384.72,0.0,0.0,-1864.28,0.0,0
2,1,181.00,181.0,0.00,0.0,0.0,-181.00,0.0,1
3,1,181.00,181.0,0.00,21182.0,0.0,-181.00,-21182.0,1
4,1,11668.14,41554.0,29885.86,0.0,0.0,-11668.14,0.0,0


## Train Test Split

A divisão dos dados foi feita da seguinte maneira:
- Foi separada a classe de interesse (`isFraud`) como **y** e as demais como **X**
- A divisão foi feita em uma proporção 70-30 devido ao elevado volume de dados
- Foi utilizado o hiperparâmetro `stratify` de modo a se manter a proporção de classes positivas e negativas tanto para o test set quanto para o training set
- Como não se tem valores categórigos, apenas foi necessário se fazer os `StandardScaler`

In [4]:
y = df_model["isFraud"]
X = df_model.drop("isFraud", axis = 1)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, stratify=y, random_state = 42)

In [6]:
scaler =  StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

`StratifiedKFold` é utilizado dividindo os dados em K partes (folds) para fazer validação cruzada, mantendo a proporção das classes em cada fold.
Ou seja, em cada Cross-Validation é garantido que cada fold tenha representatividade das classes.

In [7]:
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Fine Tunning

In [8]:
MODEL_EVALUATION_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "f1",
    "precision",
    "recall",
    "roc_auc",
    "average_precision",
    "neg_brier_score",
    "f1_weighted",
]

In [11]:
# Model Instance
xgb = XGBClassifier(use_label_encoder=False, 
                    eval_metric='logloss', 
                    random_state=42)

# Hyperparameters Dict
param_grid = {
    'n_estimators': [350, 400],
    'max_depth': [9, 12],
    'learning_rate': [0.1, 0.3],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.9, 1]
}

In [12]:
# Instância do GridSearch
grid_search = GridSearchCV(
    estimator = xgb,
    param_grid = param_grid,
    scoring = MODEL_EVALUATION_METRICS,
    refit="average_precision",
    cv = stratified_kfold,
    n_jobs = 1,
    verbose = 2
)

In [19]:
# Ajustando o modelo com os dados de treino
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    
    grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 32 candidates, totalling 160 fits
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.8; total time=  54.8s
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.8; total time=  59.0s
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.8; total time=  56.7s
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.8; total time=  58.2s
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.8; total time=  59.0s
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.9; total time= 1.1min
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.9; total time= 1.1min
[CV] END colsample_bytree=0.9, learning_rate=0.1, max_depth=9, n_estimators=350, subsample=0.9; total time=  57.2s
[CV] END colsample

In [27]:
# Resultados
print("best parameters:", grid_search.best_params_)
print("best score:", grid_search.best_score_)

best parameters: {'colsample_bytree': 1, 'learning_rate': 0.1, 'max_depth': 9, 'n_estimators': 400, 'subsample': 0.9}
best score: 0.9377652170875471


In [29]:
import joblib
filename = 'best_xgb_model.joblib'

joblib.dump(grid_search.best_estimator_, filename)

['best_xgb_model.joblib']

In [31]:
def get_mean_and_std_from_grid_search_best_estimator(
    grid_search: GridSearchCV,
) -> pd.DataFrame:
    """
    Get mean and std scores for the best estimator from a GridSearchCV object.

    Parameters
    ----------
    grid_search : GridSearchCV
        Fitted GridSearchCV object.

    Returns
    -------
    pd.DataFrame
        DataFrame with mean and std scores for the best estimator
    """
    best_estimator_index = grid_search.best_index_
    cv_results = grid_search.cv_results_

    mean_scores = {}
    std_scores = {}

    for metric in grid_search.scoring:
        mean_scores[metric] = cv_results[f"mean_test_{metric}"][best_estimator_index]
        std_scores[metric] = cv_results[f"std_test_{metric}"][best_estimator_index]

    # create dataframe with mean and std scores for each metric
    # metric | mean | std
    df_mean_scores = pd.DataFrame(mean_scores, index=["score"]).T
    df_std_scores = pd.DataFrame(std_scores, index=["std"]).T
    df_mean_std_scores = pd.concat([df_mean_scores, df_std_scores], axis=1)

    return df_mean_std_scores

In [33]:
get_mean_and_std_from_grid_search_best_estimator(grid_search)

,score,std
accuracy,0.999683,0.000015
balanced_accuracy,0.907422,0.004446
f1,0.869020,0.006452
precision,0.930856,0.005120
recall,0.814921,0.008890
roc_auc,0.999287,0.000353
average_precision,0.937765,0.003374
neg_brier_score,-0.000251,0.000011
f1_weighted,0.999672,0.000016


In [35]:
grid_search.cv_results_

{'mean_fit_time': array([53.31491632, 55.37812448, 67.56757717, 54.42177854, 49.70153923,
        52.40325956, 58.80922794, 62.27135253, 57.76403174, 46.80388918,
        54.67477503, 55.03685794, 55.66709189, 53.48436708, 62.11786761,
        61.74675622, 45.68576899, 44.46430721, 62.94206338, 60.5535749 ,
        59.44342594, 57.40618362, 69.64557047, 67.9125947 , 57.81477795,
        56.05484052, 65.77690001, 62.9464437 , 61.54788632, 62.03939037,
        72.21754751, 72.32268019]),
 'std_fit_time': array([1.5466871 , 3.8700911 , 4.93757872, 4.64517745, 2.08777944,
        0.78174394, 2.14599828, 4.64325005, 3.45080288, 0.73028745,
        0.37678778, 3.9021503 , 4.83778963, 2.56622159, 0.62788901,
        0.75016062, 0.41334721, 0.60003931, 5.42561043, 1.31687321,
        0.90140927, 0.63407051, 0.42903607, 0.45847128, 1.11483304,
        0.51337026, 1.19735887, 0.72081822, 0.5035381 , 1.09017284,
        0.75339476, 0.40067121]),
 'mean_score_time': array([4.30240874, 4.79894128, 

In [37]:
grid_search.best_index_

19